# DSH ray-tracing physics checkpoints

This notebook is the human-readable physics validation layer. Each checkpoint states an analytic or conservation result before exercising the Monte Carlo pipeline. A checkpoint passes only when its assertions pass; plots are diagnostic, not substitutes for numerical closure.

Current validated stages: source fluence and decay-side observation; source-cloud-observer coordinates; native angular-distance cloud columns with Beer-Lambert closure; and exact straight-ray column integration. Automated counterparts live in `tests/`.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from jax import random

from utils.clouds import (
    build_angular_distance_cloud, cloud_from_loaded_fits,
    optical_depth_map, recovered_delta_column_cm2,
    total_column_map_cm2, transmission_map,
)
from utils.coordinates import (
    ARCSEC_TO_RAD, angular_offset_direction,
    cartesian_to_sky, sky_position_pc,
)
from utils.source import (
    build_decay_observation_window, build_variable_powerlaw_source,
    fred_outburst_flux, sample_variable_powerlaw_source,
)

print('JAX devices:', jax.devices())

## Checkpoint 1 — source fluence and observation epoch

**Physics contract:** packet weights sum to the time-integrated unabsorbed observer-equivalent photon flux, packet times follow the FRED fluence distribution, and the observation begins strictly after the outburst peak.

In [ ]:
DAY = 86_400.0
time_edges_s = np.linspace(0.0, 120.0 * DAY, 241)
peak_time_s = 10.0 * DAY
photon_flux = fred_outburst_flux(
    time_edges_s, baseline_flux=1.0e-2, peak_excess_flux=9.0e-2,
    peak_time_s=peak_time_s, rise_time_s=2.0 * DAY,
    decay_time_s=25.0 * DAY,
)
source = build_variable_powerlaw_source(
    time_edges_s, photon_flux, energy_min_kev=1.0,
    energy_max_kev=10.0, photon_index=2.0,
)
observation = build_decay_observation_window(
    start_s=45.0 * DAY, stop_s=45.0 * DAY + 28_800.0,
    outburst_peak_s=peak_time_s,
)
packets = jax.jit(
    sample_variable_powerlaw_source, static_argnames=('n_packets',)
)(random.PRNGKey(2026), source, n_packets=100_000)

exact_fluence = np.sum(photon_flux * np.diff(time_edges_s))
sampled_fluence = np.asarray(packets.weight_observer_fluence).sum()
expected_fraction = np.asarray(source.time_bin_fluence / source.total_fluence)
sampled_fraction = np.bincount(
    np.asarray(packets.time_index), minlength=expected_fraction.size
) / packets.time_index.size
assert np.isclose(sampled_fluence, exact_fluence, rtol=2e-6)
assert np.max(np.abs(sampled_fraction - expected_fraction)) < 0.01
assert float(observation.start_s) > peak_time_s
assert float(observation.stop_s) > float(observation.start_s)

fig, ax = plt.subplots(figsize=(9, 4))
centers_day = 0.5 * (time_edges_s[:-1] + time_edges_s[1:]) / DAY
ax.plot(centers_day, photon_flux, label='generic FRED photon flux')
ax.axvline(peak_time_s / DAY, color='tab:red', ls='--', label='peak')
ax.axvspan(float(observation.start_s) / DAY, float(observation.stop_s) / DAY,
           color='tab:green', alpha=0.35, label='observation')
ax.set(xlabel='Time [day]', ylabel='F_1-10keV [ph cm^-2 s^-1]')
ax.legend()
plt.show()
print(f'exact fluence:  {exact_fluence:.6e} ph cm^-2')
print(f'packet fluence: {sampled_fluence:.6e} ph cm^-2')
print('CHECKPOINT 1 PASS')

## Checkpoint 2 — physical coordinates

**Physics contract:** sightline directions are normalized; one arcsecond at 1 kpc has the expected transverse scale; and sky-to-Cartesian conversion round-trips.

In [ ]:
direction = angular_offset_direction(1.0, 0.0)
position_pc = sky_position_pc(1.0, 1.0, 0.0)
distance_kpc, x_arcsec, y_arcsec = cartesian_to_sky(position_pc)
expected_transverse_pc = 1000.0 * np.sin(ARCSEC_TO_RAD)
assert np.isclose(np.linalg.norm(np.asarray(direction)), 1.0, rtol=1e-6)
assert np.isclose(float(position_pc[1]), expected_transverse_pc, rtol=2e-6)
assert np.isclose(float(distance_kpc), 1.0, rtol=2e-6)
assert np.isclose(float(x_arcsec), 1.0, rtol=2e-6)
assert np.isclose(float(y_arcsec), 0.0, atol=1e-7)
print(f'1 arcsec at 1 kpc: {float(position_pc[1]):.9f} pc')
print('CHECKPOINT 2 PASS')

## Checkpoint 3 — cloud column and optical-depth closure

**Physics contract:** converting a voxel column with `n_H = delta_NH / delta_r` and multiplying back by `delta_r` recovers every voxel. Integrated optical depth obeys `tau = sigma_H * sum(delta_NH)`, and transmission is `exp(-tau)`.

In [ ]:
x_axis = np.array([-1.0, 0.0, 1.0])
y_axis = np.array([-0.5, 0.5])
z_axis = np.array([0.25, 0.75, 1.25, 1.75])
spatial_column = np.array([[0.6, 1.0, 0.8], [0.9, 1.5, 0.7]]) * 1.0e20
radial_weights = np.array([0.5, 1.0, 2.0, 0.5])
delta_nh_cm2 = radial_weights[:, None, None] * spatial_column[None, :, :]
cloud = build_angular_distance_cloud(
    delta_nh_cm2, x_axis, y_axis, z_axis,
    source_distance_kpc=10.0, unit='cm-2',
)
recovered = recovered_delta_column_cm2(cloud)
total_nh = total_column_map_cm2(cloud)
cross_sections = jnp.array([1.0e-22, 4.0e-22])
tau = jax.jit(optical_depth_map)(cloud, cross_sections)
transmission = jax.jit(transmission_map)(cloud, cross_sections)
relative_error = np.max(np.abs(np.asarray(recovered) - delta_nh_cm2)
                        / np.maximum(delta_nh_cm2, 1.0))
expected_tau = np.asarray(cross_sections)[:, None, None] * np.asarray(total_nh)
assert relative_error < 3e-6
assert np.allclose(np.asarray(total_nh), delta_nh_cm2.sum(axis=0), rtol=3e-6)
assert np.allclose(np.asarray(tau), expected_tau, rtol=3e-6)
assert np.allclose(np.asarray(transmission), np.exp(-expected_tau), rtol=3e-6)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), constrained_layout=True)
images = [total_nh, tau[1], transmission[1]]
titles = ['N_H [cm^-2]', 'tau for sigma=4e-22 cm^2/H', 'Transmission']
for ax, image, title in zip(axes, images, titles):
    artist = ax.imshow(np.asarray(image), origin='lower')
    ax.set_title(title)
    plt.colorbar(artist, ax=ax, shrink=0.8)
plt.show()
print(f'max voxel column-closure error: {relative_error:.3e}')
print('CHECKPOINT 3 PASS')

### Optional checkpoint on a real FITS cube

Set `FITS_PATH` locally. This preserves the full native cube and checks column closure without downsampling or rescaling.

In [ ]:
FITS_PATH = None
if FITS_PATH is None:
    print('Optional real-FITS checkpoint skipped: set FITS_PATH to run it.')
else:
    from utils.fits_cube import load_cube
    loaded = load_cube(FITS_PATH)
    real_cloud = cloud_from_loaded_fits(loaded, source_distance_kpc=10.0)
    reconstructed = np.asarray(recovered_delta_column_cm2(real_cloud))
    original = np.asarray(real_cloud.delta_nh_cm2)
    closure = np.max(np.abs(reconstructed - original) / np.maximum(original, 1.0))
    assert closure < 3e-6
    print(f'real-cube maximum voxel closure error: {closure:.3e}')
    print('OPTIONAL REAL-FITS CHECKPOINT PASS')

## Checkpoint 4 — exact straight-ray column integration

**Physics contract:** a radial ray through every angular pixel must recover that pixel's native `sum(delta_NH)` exactly. A ray outside the angular field must integrate to zero, and optical depth must remain `sigma_H * integrated_column`. The algorithm intersects spherical radial boundaries and angular planes analytically; it does not use fixed spatial substeps.

In [ ]:
from utils.ray_integrals import integrate_ray_column_cm2, integrate_ray_optical_depth

measured_nh = np.empty_like(np.asarray(total_nh))
for iy, y_arcsec in enumerate(y_axis):
    for ix, x_arcsec in enumerate(x_axis):
        direction = angular_offset_direction(x_arcsec, y_arcsec)
        measured_nh[iy, ix] = integrate_ray_column_cm2(
            cloud, origin_pc=np.zeros(3), direction=direction,
            max_distance_pc=10_000.0,
        )
expected_nh = np.asarray(total_nh)
ray_relative_error = np.max(
    np.abs(measured_nh - expected_nh) / np.maximum(expected_nh, 1.0)
)
outside_column = integrate_ray_column_cm2(
    cloud, np.zeros(3), angular_offset_direction(20.0, 20.0), 10_000.0
)
test_direction = angular_offset_direction(0.0, -0.5)
test_column = integrate_ray_column_cm2(
    cloud, np.zeros(3), test_direction, 10_000.0
)
test_tau = integrate_ray_optical_depth(
    cloud, np.zeros(3), test_direction, 10_000.0, 4.0e-22
)
assert ray_relative_error < 5e-6
assert float(outside_column) == 0.0
assert np.isclose(float(test_tau), 4.0e-22 * float(test_column), rtol=2e-6)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), constrained_layout=True)
for ax, image, title in zip(
    axes, [expected_nh, measured_nh], ['Native sum(delta_NH)', 'Ray-integrated N_H']
):
    artist = ax.imshow(image, origin='lower')
    ax.set_title(title)
    plt.colorbar(artist, ax=ax, shrink=0.8)
plt.show()
print(f'max radial-ray column error: {ray_relative_error:.3e}')
print('CHECKPOINT 4 PASS')

## Planned checkpoints

| Stage | Required physics result | Status |
|---|---|---|
| 1. Source | fluence, sampled time/energy distributions, decay-side observation | PASS above |
| 2. Coordinates | normalization, angular scale, round trip | PASS above |
| 3. Cloud input | voxel/column closure and Beer-Lambert attenuation | PASS above |
| 4. Ray integration | exact uniform-slab and multi-slab optical depth | PASS above |
| 5. Interaction process | sampled free-path and scatter/absorb branching | next |
| 6. Dust kernel | cross-section normalization and angular sampling | planned |
| 7. Time delay | thin-screen delay and arrival-window selection | planned |
| 8. Camera | solid-angle and total-weight closure in images | planned |

Run `python -W error -m compileall -q -f utils tests` and `python -m unittest discover -s tests -v` after every checkpoint.